In [97]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Evaluation Metrics
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

df = pd.read_csv('../data/student_dropout_dataset_v3.csv')

In [98]:
df['Parental_Education'] = df['Parental_Education'].fillna('Unknown')
education_mapping = {
    'Unknown': 0,
    'High School': 1,
    'Bachelor': 2,
    'Master': 3,
    'PhD': 4
}

df['Parental_Education_encoded'] = df['Parental_Education'].map(education_mapping)
df = pd.get_dummies(df, columns=['Department'], drop_first=True, dtype=int)

binary_columns = ['Internet_Access', 'Part_Time_Job', 'Scholarship']
for column in binary_columns:
    df[column] = df[column].map({'Yes': 1, 'No': 0})

gender_mapping = {'Female': 1, 'Male': 0}
df['Gender_encoded'] = df['Gender'].map(gender_mapping)

semester_mapping = {
    'Year 1': 1,
    'Year 2': 2,
    'Year 3': 3,
    'Year 4': 4
}
df['Semester_encoded'] = df['Semester'].map(semester_mapping)

In [99]:
y = df['Dropout']
X = df.drop('Dropout', axis=1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [100]:
def df_preprocess(df):
    # Group by Parental_Education, calculate the median for each group, and fill the blanks
    df['Family_Income'] = df.groupby('Parental_Education')['Family_Income'].transform(
        lambda x: x.fillna(x.median())
    )
    df['Study_Hours_per_Day'] = df.groupby('Parental_Education')['Study_Hours_per_Day'].transform(
        lambda x: x.fillna(x.mean())
    )
    df['Stress_Index'] = df['Stress_Index'].fillna(df['Stress_Index'].median())
    df['GPA_trend'] = df["CGPA"] - df['Semester_GPA']
    # Create an Age_Gap feature based on the dataset's universal mean
    df['Age_Gap'] = df['Age'] - 21.0 # average age
    columns_to_drop = ['Student_ID','Gender', 'Semester', 'Parental_Education', 'GPA', 'Semester_GPA', 'Age']
    df = df.drop(columns_to_drop, axis=1)
    return df

In [101]:
X_train = df_preprocess(X_train)
X_test = df_preprocess(X_test)

columns_gpa = ['CGPA', 'GPA_trend']
X_train_wo_gpa = X_train.drop(columns_gpa, axis=1)
X_test_wo_gpa = X_test.drop(columns_gpa, axis=1)

In [102]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_wo_gpa = scaler.fit_transform(X_train_wo_gpa)
X_test_scaled_wo_gpa = scaler.transform(X_test_wo_gpa)

X_train_arr = X_train.values
X_test_arr = X_test.values

X_train_arr_wo_gpa = X_train_wo_gpa.values
X_test_arr_wo_gpa = X_test_wo_gpa.values

In [103]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        class_weight='balanced',   # handles class imbalance
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),
}

scaled_models = {'Logistic Regression'}

In [104]:
results = {}

for name, model in models.items():
    X_tr = X_train_scaled if name in scaled_models else X_train_arr
    X_te = X_test_scaled if name in scaled_models else X_test_arr

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'report': classification_report(y_test, y_pred),
        'y_pred': y_pred,
        'y_proba': y_proba,
    }

    print(f"\n{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {results[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {results[name]['roc_auc']:.4f}")
    print(f"\n{results[name]['report']}")


  Logistic Regression
  Accuracy : 0.7365
  ROC-AUC  : 0.8151

              precision    recall  f1-score   support

           0       0.91      0.73      0.81      1529
           1       0.46      0.76      0.58       471

    accuracy                           0.74      2000
   macro avg       0.69      0.74      0.69      2000
weighted avg       0.80      0.74      0.75      2000


  Random Forest
  Accuracy : 0.7635
  ROC-AUC  : 0.8028

              precision    recall  f1-score   support

           0       0.88      0.81      0.84      1529
           1       0.50      0.63      0.56       471

    accuracy                           0.76      2000
   macro avg       0.69      0.72      0.70      2000
weighted avg       0.79      0.76      0.77      2000


  Gradient Boosting
  Accuracy : 0.7970
  ROC-AUC  : 0.8030

              precision    recall  f1-score   support

           0       0.83      0.93      0.87      1529
           1       0.61      0.38      0.47       471

In [105]:
results_wo_gpa = {}

for name, model in models.items():
    X_tr = X_train_scaled_wo_gpa if name in scaled_models else X_train_arr_wo_gpa
    X_te = X_test_scaled_wo_gpa if name in scaled_models else X_test_arr_wo_gpa

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results_wo_gpa[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'report': classification_report(y_test, y_pred),
        'y_pred': y_pred,
        'y_proba': y_proba,
    }

    print(f"\n{'=' * 50}")
    print(f"  {name}")
    print(f"{'=' * 50}")
    print(f"  Accuracy : {results_wo_gpa[name]['accuracy']:.4f}")
    print(f"  ROC-AUC  : {results_wo_gpa[name]['roc_auc']:.4f}")
    print(f"\n{results_wo_gpa[name]['report']}")


  Logistic Regression
  Accuracy : 0.6525
  ROC-AUC  : 0.7203

              precision    recall  f1-score   support

           0       0.86      0.65      0.74      1529
           1       0.37      0.65      0.47       471

    accuracy                           0.65      2000
   macro avg       0.61      0.65      0.60      2000
weighted avg       0.74      0.65      0.68      2000


  Random Forest
  Accuracy : 0.7255
  ROC-AUC  : 0.7041

              precision    recall  f1-score   support

           0       0.82      0.82      0.82      1529
           1       0.42      0.42      0.42       471

    accuracy                           0.73      2000
   macro avg       0.62      0.62      0.62      2000
weighted avg       0.73      0.73      0.73      2000


  Gradient Boosting
  Accuracy : 0.7715
  ROC-AUC  : 0.7031

              precision    recall  f1-score   support

           0       0.79      0.96      0.87      1529
           1       0.56      0.15      0.23       471